In [5]:
import pypsa
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
%matplotlib inline

from seersandbox import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Get cost data

In [137]:
YEAR = 2025
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{YEAR}.csv"
costs = pd.read_csv(url, index_col=[0, 1])
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs = costs.value.unstack().fillna({"discount rate": 0.07, "lifetime": 20, "FOM": 0})

In [138]:
costs

parameter,Bottom storage temperature,C in fuel,C stored,CO2 intensity,CO2 stored,FOM,Motor size,Top storage temperature,VOM,ammonia-input,...,oil-input,ore-input,p_nom_ratio,pelletizing cost,scrap-input,slag-input,standing losses,temperature difference,wood-input,yield-biochar
technology,,,,,,,,,,,,,,,,,,,,,
Alkaline electrolyzer large size,NaN,NaN,NaN,NaN,NaN,4.6000,NaN,NaN,0.2389,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Alkaline electrolyzer medium size,NaN,NaN,NaN,NaN,NaN,18.1000,NaN,NaN,0.2389,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Alkaline electrolyzer small size,NaN,NaN,NaN,NaN,NaN,18.1000,NaN,NaN,0.6426,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ammonia cracker,NaN,NaN,NaN,NaN,NaN,4.3000,NaN,NaN,NaN,1.46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BEV Bus city,NaN,NaN,NaN,NaN,NaN,0.0002,320.6897,NaN,0.0952,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
uranium,NaN,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
waste CHP,NaN,NaN,NaN,NaN,NaN,2.3789,NaN,NaN,28.4644,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
waste CHP CC,NaN,NaN,NaN,NaN,NaN,2.3789,NaN,NaN,28.4644,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Create data for load and generation

In [124]:
load_ts = construct_load_ts()
re_ts = construct_re_ts()

## Create network

In [6]:
model = pypsa.Network()

In [7]:
# Key dimensions of optimization
nzones = 3
nyears = 4
rez = 1 # resolution in hours


# Other user specified inputs
year_step = 1
start_year = 2025


# Derived inputs
years = [y for y in range(start_year, start_year + nyears*year_step, year_step)]

In [4]:
set_snapshots(model, years, rez)
set_obj_weights(model, years)

# add existing solar in each zone
# add existing wind in each zone
# add existing gas in each zone

# add build options
# solar
# wind
# gas

In [13]:
model.buses.index

RangeIndex(start=0, stop=0, step=1, name='name')

In [127]:
nzones = 10
zones = [i for i in range(nzones)]

nyears = 10
start_year = 2025
end_year = start_year + nyears

freq = "h"


growth_rate = 0.05
load_growth = 1.05**nyears
# one demand trajectory across all zones
peak = pd.DataFrame(index = pd.date_range(start=str(start_year), end=str(end_year), freq="YE"),
                    data=np.arange(1, 1 + load_growth, load_growth/float(nyears-1)),
                    columns=["peak_mw"])


# starting demand in each zone:
avg_demand = 1000
demand_values = np.random.uniform(0.9*avg_demand, 1.1*avg_demand, size=nzones)
demand = pd.DataFrame(index = zones, data=demand_values)

# starting generation capacity in each zone
existing_gen = pd.DataFrame(index = zones, columns=["gas", "pv", "wind", "battery"])
gen_avgs = pd.DataFrame(index=["gas", "pv", "wind", "battery"], columns=["mw"], data=[800, 300, 100, 100])
for gen_type in existing_gen.columns:
    existing_gen[gen_type] = np.random.uniform(0.9*gen_avgs.loc[gen_type]["mw"], 1.1*gen_avgs.loc[gen_type]["mw"], size=nzones)
    

In [94]:
sys = pypsa.Network()

for zone in zones:
    
    # Add the bus
    sys.add("Bus", str(zone), carrier="electricity")
    
    # Add bus load
    load_ts = construct_load_ts(annual_peak_mw=demand.loc[zone][0]*peak)
    sys.add("Load", str(zone) + " - demand", p_set = load_ts.load_mw)
    
    # Add the existing generators
    for gen in existing_gen.columns:
        if gen in ["pv", "wind"]:
            sys.add("Generator", str(zone) + " - " + str(gen), bus=str(zone),
                    p_max_pu = re_ts[gen + "_pu"],
                    p_nom  = existing_gen.loc[zone][gen],
                    
                   )
        elif gen in ["gas"]:
            sys.add("Generator", str(zone) + " - " + str(gen), bus=str(zone),
                    p_nom = existing_gen.loc[zone][gen],
                    
            

In [135]:
existing_gen

,gas,pv,wind,battery
0,807.354826,295.997412,99.619545,107.139855
1,743.599112,291.242583,98.797466,90.581279
2,830.924558,274.056634,100.708939,101.999925
3,824.633825,312.446527,96.860250,100.926024
4,848.157587,299.038389,100.170061,98.203167
5,751.500762,319.527308,96.072538,108.574140
6,867.329364,290.162009,91.018857,96.410241
7,754.983184,313.750848,103.366224,95.238469
8,777.119364,320.982734,103.398417,101.169922
9,879.773096,314.455218,107.709166,93.575755


In [133]:
zone = zones[0]
load_ts = construct_load_ts(annual_peak_mw=demand.loc[zone][0]*peak)

### Add buses

In [6]:
# number of nodes
n = 3
nodes = ["Node" + str(i) for i in range(n)]
sys.add("Bus", nodes)

In [7]:
sys.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
name,,,,,,,,,,,,,
Node0,1.0,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
Node1,1.0,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
Node2,1.0,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,


In [8]:
RESOLUTION = 3  # hours
url = "https://tubcloud.tu-berlin.de/s/9toBssWEdaLgHzq/download/time-series.csv"
ts = pd.read_csv(url, index_col=0, parse_dates=True)[::RESOLUTION]
ts.head()

,load_mw,pv_pu,wind_pu
timestamp,,,
2019-01-01 00:00:00,5719.26,0.000,0.1846
2019-01-01 03:00:00,5474.74,0.000,0.3146
2019-01-01 06:00:00,5413.39,0.000,0.4957
2019-01-01 09:00:00,5891.23,0.059,0.6824
2019-01-01 12:00:00,6662.42,0.063,0.7519


In [10]:
ts.tail()

,load_mw,pv_pu,wind_pu
timestamp,,,
2019-12-31 09:00:00,7029.73,0.195,0.3049
2019-12-31 12:00:00,7342.83,0.135,0.3944
2019-12-31 15:00:00,7267.22,0.000,0.3788
2019-12-31 18:00:00,6821.12,0.000,0.3157
2019-12-31 21:00:00,6191.86,0.000,0.2636
